In [3]:
import pandas as pd
import numpy as np
import nltk
import gdown
import re
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import stanza
import joblib
import dateparser
import gzip
import shutil
import pickle
from IPython.display import display, Markdown
from collections import Counter
from itertools import islice
from math import log
from gensim.models import KeyedVectors
from nltk.tokenize import word_tokenize
from scipy.stats import chi2_contingency
from scipy.sparse import vstack, hstack
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.models import Sequential, save_model, load_model
from tensorflow.keras.layers import LSTM, Dense, Embedding, Dropout, SpatialDropout1D, BatchNormalization, Bidirectional
from keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.svm import LinearSVC, SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from wordcloud import WordCloud
nltk.download("stopwords")
from nltk.corpus import stopwords
nltk.download('punkt_tab', quiet = True)
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
from collections import Counter


[nltk_data] Downloading package stopwords to /Users/OG1/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/OG1/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [4]:
import pandas as pd
import numpy as np

# Load datasets (same as in eda_aniekan notebook)
fake = pd.read_csv("../data/Fake.csv")
true = pd.read_csv("../data/True.csv")

# Add labels (0 = Fake, 1 = Real)
fake["label"] = 0
true["label"] = 1

df = pd.concat([fake, true], ignore_index=True)


# Data cleaning
word_to_remove = '(Reuters)'
df['text'] =  df['text'].str.replace(r'[\(\-–\s]*Reuters[\)\s]*', '', case=False, regex=True)
df['text'] = df['text'].str.replace(r'^[A-Z]+(?:\s+[A-Z]+)*-\s*', '', regex=True)

# Remove stopwords
stop_words = stopwords.words("english")
df["title"] = df["title"].apply(
    lambda x: " ".join(word for word in x.split() if word not in stop_words))
df["text"] = df["text"].apply(
    lambda x: " ".join(word for word in x.split() if word not in stop_words))

# Lemmatization
wnl = WordNetLemmatizer()
df["text"] = df["text"].apply(lambda x: " ".join(wnl.lemmatize(word) for word in x.split()))
df["title"] = df["title"].apply(lambda x: " ".join(wnl.lemmatize(word) for word in x.split()))
print(df.head())
# Drop Subject
df = df.drop(columns='subject')
df

                                               title  \
0  Donald Trump Sends Out Embarrassing New Year’s...   
1  Drunk Bragging Trump Staffer Started Russian C...   
2  Sheriff David Clarke Becomes An Internet Joke ...   
3  Trump Is So Obsessed He Even Has Obama’s Name ...   
4  Pope Francis Just Called Out Donald Trump Duri...   

                                                text subject  \
0  Donald Trump wish Americans Happy New Year lea...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, revealed former Milwaukee Sheriff D...    News   
3  On Christmas day, Donald Trump announced would...    News   
4  Pope Francis used annual Christmas Day message...    News   

                date  label  
0  December 31, 2017      0  
1  December 31, 2017      0  
2  December 30, 2017      0  
3  December 29, 2017      0  
4  December 25, 2017      0  


,title,text,date,label
0,Donald Trump Sends Out Embarrassing New Year’s...,Donald Trump wish Americans Happy New Year lea...,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian C...,House Intelligence Committee Chairman Devin Nu...,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke ...,"On Friday, revealed former Milwaukee Sheriff D...","December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name ...,"On Christmas day, Donald Trump announced would...","December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Duri...,Pope Francis used annual Christmas Day message...,"December 25, 2017",0
...,...,...,...,...
44893,'Fully committed' NATO back new U.S. approach ...,NATO ally Tuesday welcomed President Donald Tr...,"August 22, 2017",1
44894,LexisNexis withdrew two product Chinese market,"LexisNexis, provider legal, regulatory busines...","August 22, 2017",1
44895,Minsk cultural hub becomes authority,"In shadow disused Soviet-era factory Minsk, st...","August 22, 2017",1
44896,Vatican upbeat possibility Pope Francis visiti...,Vatican Secretary State Cardinal Pietro Paroli...,"August 22, 2017",1


In [5]:
# LSTM model with tokenization - Arif

# Combine title and text for context
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.regularizers import l2
df['combined_text'] = df['title'].astype(str) + ' ' + df['text'].astype(str)

X_text = df['combined_text'].values
y = df['label'].values

print(f"Number of samples: {len(X_text)}")
print(f"Sample text: {X_text[0][:100]}...")  # show first 100 chars

# Step 2: Tokenize text to numbers
vocab_size = 10000  # maximum number of words to keep
tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(X_text)

# Convert texts to sequences of numbers
X_sequences = tokenizer.texts_to_sequences(X_text)

print(f"Sample sequence: {X_sequences[0][:20]}...")  # show first 20 tokens

# Step 3: Pad sequences to same length
max_len = 80
X_padded = pad_sequences(X_sequences, maxlen=max_len, padding='post', truncating='post')

print(f"Padded shape: {X_padded.shape}")

# Step 4: Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_padded, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

# Step 5: Build LSTM model
model = Sequential([
    Embedding(vocab_size, 128, input_length=max_len, input_shape=(max_len,)),
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(128, dropout=0.2, recurrent_dropout=0.2, kernel_regularizer=l2(0.01))),
    Dense(8, activation='elu', kernel_regularizer=l2(0.01)),
    Dropout(0.5),
    BatchNormalization(
        axis=-1, 
        momentum=0.99, 
        epsilon=0.001, 
        center=True, 
        scale=True, 
        beta_initializer='zeros', 
        gamma_initializer='ones',
        moving_mean_initializer='zeros', 
        moving_variance_initializer='ones'  # fixed typo
    ),
    Dense(1, activation='sigmoid')
])


model.compile(
    optimizer=AdamW(learning_rate=5e-5, weight_decay=5e-4, clipnorm=1.0),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\nModel Architecture:")
model.summary()

# Step 6: Train
print("\nTraining model...")
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=16,
    callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
    verbose=1
)

#Step 7:
#Saving trained tokenizer to be used later when model testing
joblib.dump(tokenizer, "tokenizer.pkl")

# Step 8: Evaluate
print("\nEvaluating on test set...")
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Loss: {loss:.4f}")

# Step 9: Make predictions
predictions = model.predict(X_test)
pred_classes = (predictions > 0.5).astype(int).flatten()

# Step 10: Show detailed results
print("\n" + "="*50)
print("RESULTS")
print("="*50)
print("\nClassification Report:")
print(classification_report(y_test, pred_classes))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pred_classes))

model.save('fake_news_lstm_model.keras')

joblib.dump(tokenizer, 'tokenizer.pkl')

print("Model and tokenizer saved successfully!")


Number of samples: 44898
Sample text: Donald Trump Sends Out Embarrassing New Year’s Eve Message; This Disturbing Donald Trump wish Americ...
Sample sequence: [21, 2, 4845, 340, 2688, 17, 3611, 4399, 495, 38, 2758, 21, 2, 1986, 161, 1627, 17, 15, 629, 53]...
Padded shape: (44898, 80)
Training samples: 35918
Test samples: 8980

Model Architecture:


/Users/OG1/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
/Users/OG1/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 80, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 80, 128)        │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 256)            │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8)              │         2,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 8)              │            32 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,545,265 (5.89 MB)

 Trainable params: 1,545,249 (5.89 MB)

 Non-trainable params: 16 (64.00 B)


Training model...
Epoch 1/5
1796/1796 ━━━━━━━━━━━━━━━━━━━━ 84s 45ms/step - accuracy: 0.7666 - loss: 3.2204 - val_accuracy: 0.9932 - val_loss: 0.7585
Epoch 2/5
1796/1796 ━━━━━━━━━━━━━━━━━━━━ 79s 44ms/step - accuracy: 0.9535 - loss: 0.6424 - val_accuracy: 0.9937 - val_loss: 0.2306
Epoch 3/5
1796/1796 ━━━━━━━━━━━━━━━━━━━━ 83s 46ms/step - accuracy: 0.9572 - loss: 0.2633 - val_accuracy: 0.9944 - val_loss: 0.1031
Epoch 4/5
1796/1796 ━━━━━━━━━━━━━━━━━━━━ 91s 51ms/step - accuracy: 0.9663 - loss: 0.1570 - val_accuracy: 0.9950 - val_loss: 0.0657
Epoch 5/5
1796/1796 ━━━━━━━━━━━━━━━━━━━━ 91s 51ms/step - accuracy: 0.9672 - loss: 0.1257 - val_accuracy: 0.9957 - val_loss: 0.0438

Evaluating on test set...
281/281 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9951 - loss: 0.0459
Test Accuracy: 0.9958
Test Loss: 0.0448
281/281 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step

RESULTS

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4696

In [6]:

new_fake_and_true = pd.read_csv("../data/Fake_Real_News_Data.csv")

if 'Unnamed: 0' in new_fake_and_true.columns:
    new_fake_and_true = new_fake_and_true.drop(columns='Unnamed: 0')

new_fake_and_true.head()

label_map = {'FAKE': 0, 'REAL': 1}
new_fake_and_true['label'] = new_fake_and_true['label'].map(label_map)

#remove stopwords 
# Remove stopwords
stop_words = stopwords.words("english")
new_fake_and_true["title"] = new_fake_and_true["title"].apply(
    lambda x: " ".join(word for word in x.split() if word not in stop_words))
new_fake_and_true["text"] = new_fake_and_true["text"].apply(
    lambda x: " ".join(word for word in x.split() if word not in stop_words))

# Lemmatization
wnl = WordNetLemmatizer()
new_fake_and_true["text"] = new_fake_and_true["text"].apply(lambda x: " ".join(wnl.lemmatize(word) for word in x.split()))
new_fake_and_true["title"] = new_fake_and_true["title"].apply(lambda x: " ".join(wnl.lemmatize(word) for word in x.split()))
print(new_fake_and_true.head())
new_fake_and_true

FileNotFoundError: [Errno 2] No such file or directory: '../data/Fake_Real_News_Data.csv'

In [ ]:
welfake_data = pd.read_csv("../data/WELFake_Dataset.csv")
welfake_data.head()

if "Unnamed: 0" in welfake_data.columns:
    welfake_data = welfake_data.drop(columns='Unnamed: 0')
# Remove stopwords
stop_words = stopwords.words("english")
welfake_data["title"] = welfake_data["title"].apply(
    lambda x: " ".join(word for word in str(x).split() if word not in stop_words) if pd.notna(x) else "")
welfake_data["text"] = welfake_data["text"].apply(
    lambda x: " ".join(word for word in str(x).split() if word not in stop_words) if pd.notna(x) else "")

# Lemmatization
wnl = WordNetLemmatizer()
welfake_data["text"] = welfake_data["text"].apply(lambda x: " ".join(wnl.lemmatize(word) for word in str(x).split()) if pd.notna(x) else "")
welfake_data["title"] = welfake_data["title"].apply(lambda x: " ".join(wnl.lemmatize(word) for word in str(x).split()) if pd.notna(x) else "")
print(welfake_data.head())
welfake_data.head()
welfake_data.tail()

In [ ]:
#Testing the LSTM Model


#Grabbing the columns I want to predict on
#Grabbing both text and title columns
dataset1 = (new_fake_and_true['title'] + ' ' + new_fake_and_true['text']).astype(str).tolist()

#loading the original tokenizer

tokenizer = joblib.load("tokenizer.pkl")
#Building the vocabulary by fitting the tokenizer on title_and_text data

#converting text to sequences of integers
sequences = tokenizer.texts_to_sequences(dataset1)

#Padding sequences

X_new = pad_sequences(sequences, maxlen=80, padding="post")

y_pred = new_fake_and_true['label'].values





y_new_prob= model.predict(X_new)

predictions = (y_new_prob > 0.5).astype(int).flatten()




print(f"Accuracy score on new dataset: ", accuracy_score(predictions, y_pred))
# print(classification_report(y_pred, y_new_pred))

In [ ]:
dataset2 = (welfake_data['title'] + ' ' + welfake_data['text']).astype(str).tolist()


#loading original tokenizer from model training
tokenizer = joblib.load("tokenizer.pkl")
sequences = tokenizer.texts_to_sequences(dataset2)



X_original= pad_sequences(sequences, maxlen= 80, padding="post")

y_original = welfake_data['label'].values

predictions = model.predict(X_original)

modified_predictions = (predictions > 0.5).astype(int).flatten()

print(f"Accuracy score on dataset2: ", accuracy_score(modified_predictions, y_original))